# Getting started with GA and SC voting data

First we explore General Assembly (GA) data, reshape it, and reproduce Policy Pulse's
agreement calculation. Then you apply the same ideas to Security Council (SC) data.

**What you will do**
1. Inspect country-level GA votes and transform them into one row per resolution.
2. Calculate agreement on one resolution.
3. Reuse that function in a loop to compare two countries across resolutions.
4. Inspect SC votes and complete the transformation and analysis exercises.

Run the worked examples from top to bottom. The SC exercise cells contain comments for you
to replace with your code; they intentionally have no solutions.

This notebook uses small, historical CSV samples already in this repository. It requires
Python, pandas, NumPy, and a Jupyter kernel, **without a database, Docker, or live downloads**.
Select the project's Python environment in your notebook editor. These are legacy export
samples; the current app reads normalized tables from PostgreSQL.


In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the repository whether Jupyter starts here or at the repository root.
repo_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "notebooks/data/GA/2025_7_23_ga_voting_sample.csv").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from inside the Policy Pulse repository.")

data_dir = repo_root / "notebooks" / "data"
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 70)


## 1. Inspect the GA data

In the original **long format**, one row is one country's vote on one resolution.
`undl_id` identifies the resolution record, `ms_code` identifies the country, and
`ms_vote` contains its vote. `ms_name` is the name recorded in the source.

Vote codes: **Y** = Yes, **N** = No, **A** = Abstain, **X** = Non-voting.
A missing value is also excluded from our agreement calculations; it is not a No vote.


In [2]:
ga_raw = pd.read_csv(
    data_dir / "GA" / "2025_7_23_ga_voting_sample.csv",
    dtype={"undl_id": "string", "ms_code": "string"},
    parse_dates=["date"],
)

print(f"{len(ga_raw):,} country-vote rows; {ga_raw['undl_id'].nunique()} resolution(s)")
display(ga_raw.head())


100 country-vote rows; 1 resolution(s)


,undl_id,ms_code,ms_name,ms_vote,date,session,...,total_yes,total_no,total_abstentions,total_non_voting,total_ms,undl_link
0,279803,AFG,AFGHANISTAN,Y,1985-12-12,40,...,142.0,0.0,6.0,11.0,159.0,https://digitallibrary.un.org/record/279803
1,279803,ALB,ALBANIA,X,1985-12-12,40,...,142.0,0.0,6.0,11.0,159.0,https://digitallibrary.un.org/record/279803
2,279803,DZA,ALGERIA,Y,1985-12-12,40,...,142.0,0.0,6.0,11.0,159.0,https://digitallibrary.un.org/record/279803
3,279803,AGO,ANGOLA,Y,1985-12-12,40,...,142.0,0.0,6.0,11.0,159.0,https://digitallibrary.un.org/record/279803
4,279803,ATG,ANTIGUA AND BARBUDA,Y,1985-12-12,40,...,142.0,0.0,6.0,11.0,159.0,https://digitallibrary.un.org/record/279803


**Check the size of the sample before interpreting it.** This file contains 100 country-vote
rows from just one resolution, whose source reports 159 members. It is a partial extract.
The totals describe the full source record; the rows here cover only part of it.


In [3]:
ga_coverage = ga_raw.groupby("undl_id").agg(
    resolution=("resolution", "first"),
    countries_in_sample=("ms_code", "nunique"),
    reported_members=("total_ms", "first"),
)
display(ga_coverage)
display(ga_raw["ms_vote"].value_counts(dropna=False).rename("rows").to_frame())


,resolution,countries_in_sample,reported_members
undl_id,,,
279803,A/RES/40/86,100,159.0


,rows
ms_vote,
Y,88
X,8
A,4


## 2. Transform the GA data

Our analysis uses a **wide format**: one row per resolution, with a vote column for each
country. Keep the country codes as column names and retain resolution metadata separately.

We check that each `(undl_id, ms_code)` pair is unique before using `pivot`. We do not average
or silently discard duplicate votes. Pivoting only on the identifier also avoids dropping
records because an optional metadata field is missing.


In [4]:
assert not ga_raw.duplicated(["undl_id", "ms_code"]).any(), "Check duplicate votes first."

ga_metadata_columns = [
    "undl_id", "date", "resolution", "session", "title",
    "total_yes", "total_no", "total_abstentions", "total_non_voting", "total_ms",
]
ga_metadata = ga_raw[ga_metadata_columns].drop_duplicates()
assert ga_metadata["undl_id"].is_unique, "Resolve conflicting metadata before reshaping."

ga_votes = ga_raw.pivot(index="undl_id", columns="ms_code", values="ms_vote")
ga_votes.columns.name = None
ga_country_columns = ga_votes.columns.tolist()

ga_wide = ga_metadata.merge(ga_votes, on="undl_id", how="left", validate="one_to_one")

print(f"Before: {ga_raw.shape}; after: {ga_wide.shape}")
display(ga_wide.head())


Before: (100, 20); after: (1, 110)


,undl_id,date,resolution,session,title,total_yes,...,MWI,MYS,NPL,SLV,TCD,YMD
0,279803,1985-12-12,A/RES/40/86,40,Conclusion of effective international arrangements to assure non-n...,142.0,...,Y,Y,Y,Y,Y,Y


The single row is expected: the input contains only one resolution. Here are a few country
columns so you can see the result without scrolling through all the metadata.


In [5]:
display(ga_wide[["undl_id", "resolution", "CHN", "FRA", "IND", "FIN"]].head())

# Verify that reshaping preserved every country-vote row.
assert ga_wide[ga_country_columns].notna().sum().sum() == len(ga_raw)


,undl_id,resolution,CHN,FRA,IND,FIN
0,279803,A/RES/40/86,Y,Y,A,X


## 3. Agreement on one resolution

Policy Pulse encodes **Yes = 1**, **Abstain = 0**, and **No = −1**. For two countries:

\[
\operatorname{agreement}(a,b) = 1 - \frac{|v_a - v_b|}{2}
\]

| Votes | Agreement |
|---|---:|
| Same valid vote, including two abstentions | 1.0 |
| Yes / Abstain or No / Abstain | 0.5 |
| Yes / No | 0.0 |
| Either vote is X or missing | NaN — excluded |

Giving abstention partial agreement is a methodological choice. It is not the same as
counting only exact matches. The readable function below implements the same pairwise
formula as [`DataProcessor`](../../app/un_data_stream/data/processor.py), without importing
the app or connecting to its database.


In [6]:
def agreement_for_resolution(resolution_row, country_a, country_b):
    '''Compare two countries on one resolution; return NaN if either did not vote.'''
    vote_values = {"Y": 1.0, "A": 0.0, "N": -1.0}
    vote_a = vote_values.get(resolution_row[country_a], np.nan)
    vote_b = vote_values.get(resolution_row[country_b], np.nan)

    if pd.isna(vote_a) or pd.isna(vote_b):
        return np.nan

    return 1.0 - abs(vote_a - vote_b) / 2.0


In [7]:
one_resolution = ga_wide.iloc[0]
print(one_resolution["resolution"])

# Predict each result before running this cell.
examples = []
for country_a, country_b in [("CHN", "FRA"), ("CHN", "IND"), ("CHN", "FIN")]:
    examples.append({
        "country_a": country_a,
        "vote_a": one_resolution[country_a],
        "country_b": country_b,
        "vote_b": one_resolution[country_b],
        "agreement": agreement_for_resolution(one_resolution, country_a, country_b),
    })

display(pd.DataFrame(examples))


A/RES/40/86


,country_a,vote_a,country_b,vote_b,agreement
0,CHN,Y,FRA,Y,1.0
1,CHN,Y,IND,A,0.5
2,CHN,Y,FIN,X,NaN


### One score for the whole resolution: consensus

To summarize the resolution, average agreement across all distinct country pairs with valid
votes. Exclude self-comparisons, count each pair once, and exclude pairs involving X or missing
values. `combinations` gives us those distinct pairs.

**Here this is consensus among the sampled voters, not the full resolution's consensus.**


In [8]:
pair_agreements = []
for country_a, country_b in combinations(ga_country_columns, 2):
    score = agreement_for_resolution(one_resolution, country_a, country_b)
    if pd.notna(score):
        pair_agreements.append(score)

sample_consensus = np.mean(pair_agreements) if pair_agreements else np.nan
print(f"Comparable country pairs: {len(pair_agreements):,}")
print(f"Consensus among sampled voters: {sample_consensus:.3f}")


Comparable country pairs: 4,186
Consensus among sampled voters: 0.958


## 4. Agreement between two countries across resolutions

We now need more than one resolution. Load the separate, already-transformed **100-resolution
GA sample**. This file is a different extract: it is **not** the result of the pivot above.
Its country columns use historical **names**, whereas our pivot uses country **codes**.
We use its labels directly and do not silently merge historical names.

The example compares `CHINA` and `FRANCE`. The same function works because its arguments
are simply the two column labels. These 100 resolutions are a teaching sample, not a
representative measure of the countries' agreement over all history.


In [9]:
ga_history = pd.read_csv(
    data_dir / "GA" / "transformed_ga_votes_sample.csv",
    dtype={"undl_id": "string"},
    parse_dates=["date"],
)

country_a = "CHINA"
country_b = "FRANCE"
assert country_a != country_b, "Choose two different countries."
assert {country_a, country_b}.issubset(ga_history.columns), "Check the country labels."
assert ga_history["undl_id"].is_unique, "Expected one row per resolution."

print(f"{len(ga_history)} resolutions, {ga_history['date'].min().date()} to "
      f"{ga_history['date'].max().date()}")
display(ga_history[["undl_id", "date", "resolution", country_a, country_b]].head())


100 resolutions, 1983-10-27 to 1983-12-19


,undl_id,date,resolution,CHINA,FRANCE
0,278340,1983-10-27,A/RES/38/3,Y,Y
1,278341,1983-11-02,A/RES/38/7,Y,Y
2,278342,1983-11-10,A/RES/38/9,Y,Y
3,278343,1983-11-15,A/RES/38/11,Y,Y
4,278344,1983-11-16,A/RES/38/12,Y,A


Loop over the resolutions and reuse `agreement_for_resolution` from the previous section.
Keep the per-resolution results so that the average is easy to inspect.


In [10]:
results = []
for _, resolution_row in ga_history.iterrows():
    results.append({
        "undl_id": resolution_row["undl_id"],
        "date": resolution_row["date"],
        "resolution": resolution_row["resolution"],
        "vote_a": resolution_row[country_a],
        "vote_b": resolution_row[country_b],
        "agreement": agreement_for_resolution(resolution_row, country_a, country_b),
    })

country_agreement = pd.DataFrame(results)
display(country_agreement.head(10))


,undl_id,date,resolution,vote_a,vote_b,agreement
0,278340,1983-10-27,A/RES/38/3,Y,Y,1.0
1,278341,1983-11-02,A/RES/38/7,Y,Y,1.0
2,278342,1983-11-10,A/RES/38/9,Y,Y,1.0
3,278343,1983-11-15,A/RES/38/11,Y,Y,1.0
4,278344,1983-11-16,A/RES/38/12,Y,A,0.5
5,278639,1983-11-21,A/RES/38/13,Y,N,0.0
6,278640,1983-11-22,A/RES/38/17,Y,N,0.0
7,278641,1983-11-22,A/RES/38/19,Y,A,0.5
8,278642,1983-11-22,A/RES/38/25,Y,Y,1.0
9,278643,1983-11-23,A/RES/38/29,Y,Y,1.0


In [11]:
# pandas excludes NaN values from the mean; they are not treated as disagreements.
comparable = country_agreement["agreement"].notna()
average_agreement = country_agreement.loc[comparable, "agreement"].mean()

display(pd.DataFrame([{
    "country_a": country_a,
    "country_b": country_b,
    "resolutions_in_sample": len(country_agreement),
    "comparable_resolutions": int(comparable.sum()),
    "excluded_resolutions": int((~comparable).sum()),
    "mean_agreement": average_agreement,
}]))


,country_a,country_b,resolutions_in_sample,comparable_resolutions,excluded_resolutions,mean_agreement
0,CHINA,FRANCE,100,96,4,0.645833


**Try it:** choose another pair, then rerun this section. Which individual resolutions drive
their disagreement? Why should we report the number of comparable resolutions alongside the
mean? What result should we report if the two countries have no comparable votes?

The app uses vectorized arrays for speed. The loop here deliberately makes the calculation
visible: score each resolution, exclude invalid comparisons, then take the mean.


## 5. Inspect the SC data

The SC input also has one country-vote row per resolution. Inspect the first rows and the
columns before copying any GA transformation code. In particular, look for `description`,
`agenda`, `modality`, and `permanent_member`.


In [12]:
sc_raw = pd.read_csv(
    data_dir / "SC" / "2025_7_21_sc_voting_sample.csv",
    dtype={"undl_id": "string", "ms_code": "string"},
    parse_dates=["date"],
)

print(f"{len(sc_raw):,} country-vote rows; {sc_raw['undl_id'].nunique()} resolution(s)")
display(sc_raw.head())


100 country-vote rows; 7 resolution(s)


,undl_id,ms_code,ms_name,permanent_member,ms_vote,date,...,total_no,total_abstentions,total_non_voting,total_ms,modality,undl_link
0,278790,CHN,CHINA,True,Y,1982-04-09,...,0,0,0,15,Vote,https://digitallibrary.un.org/record/278790
1,278790,FRA,FRANCE,True,Y,1982-04-09,...,0,0,0,15,Vote,https://digitallibrary.un.org/record/278790
2,278790,GUY,GUYANA,False,Y,1982-04-09,...,0,0,0,15,Vote,https://digitallibrary.un.org/record/278790
3,278790,IRL,IRELAND,False,Y,1982-04-09,...,0,0,0,15,Vote,https://digitallibrary.un.org/record/278790
4,278790,JPN,JAPAN,False,Y,1982-04-09,...,0,0,0,15,Vote,https://digitallibrary.un.org/record/278790


In [13]:
display(sc_raw.dtypes.rename("dtype").to_frame())
print("SC-only columns:", sorted(set(sc_raw.columns) - set(ga_raw.columns)))
print("GA-only columns:", sorted(set(ga_raw.columns) - set(sc_raw.columns)))


,dtype
undl_id,string[python]
ms_code,string[python]
ms_name,object
permanent_member,bool
ms_vote,object
date,datetime64[ns]
resolution,object
draft,object
meeting,object
description,object


SC-only columns: ['agenda', 'description', 'modality', 'permanent_member']
GA-only columns: ['agenda_title', 'committee_report', 'session', 'title']


## 6. Your turn: transform and analyze SC votes

### Task 1 — Understand the sample

- Count the rows and distinct resolutions; inspect the vote-code distribution.
- For each resolution, compare the number of distinct countries in the extract with `total_ms`.
- Identify incomplete resolutions. Do the votes in this extract reproduce the reported totals?
- Find missing metadata. Which missing values should leave a resolution in our analysis?

**Hint:** start with `groupby("undl_id").agg(...)`, `value_counts`, and `isna`.
Keep incomplete records visible in your coverage report rather than silently treating them as
complete voting records.


In [14]:
# Your code: build sc_coverage with a row for each resolution.
# Suggested columns: resolution, countries_in_sample, reported_members, complete_in_sample.

# Your code: inspect vote codes, totals, and missing fields.


### Task 2 — Decide which columns belong where

List the resolution metadata you want to retain, and separate it from country-vote fields.
For example, should `permanent_member` be a resolution-level field, or should it stay attached
to each country's vote? Does `description` play the same role as the GA `title`?

Check duplicate `(undl_id, ms_code)` pairs and conflicting metadata before reshaping.
Keep `undl_id` as the record key; do not rely on the resolution label alone.


In [15]:
# Your code: choose sc_metadata_columns and construct sc_metadata.
# Your code: check vote-pair uniqueness and metadata consistency.
# Keep country-level attributes separately rather than losing them in the pivot.


### Task 3 — Create one row per SC resolution

Use the GA example to create `sc_votes`, `sc_country_columns`, and `sc_wide`:

1. Pivot on `undl_id`, with `ms_code` as the columns and `ms_vote` as the values.
2. Join the resolution metadata back with a one-to-one validation.
3. Show `sc_wide.head()` and compare the before/after shapes.
4. Verify that you preserved the original vote rows and have one row per identifier.

**Do not fill blank country cells with `N` or `A`.** A blank may reflect non-membership,
an incomplete sample, or missing source data. It is not an observed vote. Pivot on the
identifier, not a long list of potentially missing metadata fields.


In [16]:
# Your code: pivot, join metadata, and inspect sc_wide.head().

# Your code: verify row uniqueness and preservation of the original votes.


### Task 4 — Reuse the agreement function

- Choose two countries with comparable votes; for example, inspect `CHN` and `FRA`.
- Use `agreement_for_resolution` on one SC resolution and explain the result.
- Reuse the GA loop to calculate their mean agreement across SC resolutions.
- Report the mean **and** the number of comparable resolutions.
- Repeat using only the resolutions your coverage check identifies as complete.

For resolution-level consensus, use only valid votes and distinct country pairs, as in the
GA example. Clearly label a score calculated from an incomplete resolution.


In [17]:
# Your code: calculate agreement on one SC resolution.

# Your code: loop over SC resolutions and summarize a country pair's agreement.

# Your code: compare the result with the complete-record subset.


### Task 5 — Explore further

Follow your curiosity: choose a question about the SC data and explore it using the methods
introduced here, or try an approach of your own. Add code, visualizations, and notes below
as your investigation develops.


In [ ]:
# Your exploration starts here.

## Data and method references

- [GA export documentation](../data/GA/2025_07_23_ga_voting_md.md): UN Dag Hammarskjöld
  Library, General Assembly Voting Data, version 3, July 2025.
- [SC export documentation](../data/SC/2025_07_21_sc_voting_md.md): UN Dag Hammarskjöld
  Library, Security Council Voting Data, version 3, July 2025.
- [Current agreement implementation](../../app/un_data_stream/data/processor.py).
- [Hand-calculated agreement and consensus examples in the project tests](../../tests/unit/test_resolution_matrix.py).

The export dates identify the source versions, not the date coverage of every small sample.
No live source requests are made by this notebook.
